In [0]:
import urllib.request

file_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-12.parquet"

local_path = "/tmp/yellow_tripdata_2024-12.parquet"

urllib.request.urlretrieve(file_url, local_path)

print("File downloaded successfully")

In [0]:
source_path = "/Volumes/workspace/nyc_taxi/source_data"

yellow_df = spark.read.parquet(
    f"{source_path}/Yellow/yellow_tripdata_2024-12.parquet"
)

green_df = spark.read.parquet(
    f"{source_path}/green/green_tripdata_2024-12.parquet"
)



In [0]:
yellow_df.printSchema()


In [0]:
green_df.printSchema()

In [0]:
yellow_columns = set(yellow_df.columns)
green_columns = set(green_df.columns)

print("Common columns:")
print(sorted(yellow_columns.intersection(green_columns)))

print("Yellow-only columns:")
print(sorted(yellow_columns - green_columns))

print("Green-only columns:")
print(sorted(green_columns - yellow_columns))

In [0]:
from pyspark.sql import functions as F

yellow_silver = yellow_df.select(
    F.col("VendorID").alias("vendor_id"),

    F.col("tpep_pickup_datetime").alias("pickup_datetime"),
    F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),

    F.col("PULocationID").alias("pickup_location_id"),
    F.col("DOLocationID").alias("dropoff_location_id"),

    F.col("fare_amount"),
    F.col("total_amount"),
    F.col("tip_amount"),

    F.col("Airport_fee").alias("airport_fee"),

    F.lit(None).cast("double").alias("ehail_fee"),
    F.lit(None).cast("int").alias("trip_type"),

    F.lit("yellow").alias("service_type")
)

green_silver = green_df.select(
    F.col("VendorID").alias("vendor_id"),

    F.col("lpep_pickup_datetime").alias("pickup_datetime"),
    F.col("lpep_dropoff_datetime").alias("dropoff_datetime"),

    F.col("PULocationID").alias("pickup_location_id"),
    F.col("DOLocationID").alias("dropoff_location_id"),

    F.col("fare_amount"),
    F.col("total_amount"),
    F.col("tip_amount"),

    F.lit(None).cast("double").alias("airport_fee"),

    F.col("ehail_fee"),
    F.col("trip_type"),

    F.lit("green").alias("service_type")
)

In [0]:
unified_silver = yellow_silver.unionByName(green_silver)
unified_silver.printSchema()

display(
    unified_silver.groupBy("service_type").count()
)

unified_silver.select(
    F.count("*").alias("total_records"),
    F.count("pickup_datetime").alias("valid_pickup"),
    F.count("dropoff_datetime").alias("valid_dropoff"),
    F.count("total_amount").alias("valid_total_amount"),
    F.count("pickup_location_id").alias("valid_pickup_location")
).show()

In [0]:
source_path = "/Volumes/workspace/nyc_taxi/source_data"

months = ["2024-12", "2025-01", "2025-02"]

schema_inventory = {}

for month in months:

    yellow_file = f"{source_path}/Yellow/yellow_tripdata_{month}.parquet"
    green_file = f"{source_path}/green/green_tripdata_{month}.parquet"

    yellow = spark.read.parquet(yellow_file)
    green = spark.read.parquet(green_file)

    schema_inventory[f"yellow_{month}"] = set(yellow.columns)
    schema_inventory[f"green_{month}"] = set(green.columns)

    print(f"\n========== {month} ==========")

    print("Yellow columns:", len(yellow.columns))
    print("Green columns:", len(green.columns))

    print("Yellow-only columns:")
    print(sorted(set(yellow.columns) - set(green.columns)))

    print("Green-only columns:")
    print(sorted(set(green.columns) - set(yellow.columns)))

In [0]:
for month in months:

    yellow_file = f"{source_path}/Yellow/yellow_tripdata_{month}.parquet"
    green_file = f"{source_path}/green/green_tripdata_{month}.parquet"

    yellow = spark.read.parquet(yellow_file)
    green = spark.read.parquet(green_file)

    print(f"\n{month}")

    print(
        "Yellow cbd_congestion_fee:",
        "cbd_congestion_fee" in yellow.columns
    )

    print(
        "Green cbd_congestion_fee:",
        "cbd_congestion_fee" in green.columns
    )

In [0]:
for service in ["yellow", "green"]:

    print(f"\n========== {service.upper()} ==========")

    for month in ["2024-12", "2025-01", "2025-02"]:

        folder = "Yellow" if service == "yellow" else "green"

        file_path = (
            f"{source_path}/{folder}/"
            f"{service}_tripdata_{month}.parquet"
        )

        df = spark.read.parquet(file_path)

        schema_inventory[f"{service}_{month}"] = set(df.columns)

    dec = schema_inventory[f"{service}_2024-12"]
    jan = schema_inventory[f"{service}_2025-01"]
    feb = schema_inventory[f"{service}_2025-02"]

    print("Added in January:", sorted(jan - dec))
    print("Removed in January:", sorted(dec - jan))

    print("Added in February:", sorted(feb - jan))
    print("Removed in February:", sorted(jan - feb))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.nyc_taxi_silver;


In [0]:
%sql
DESCRIBE TABLE workspace.nyc_taxi_silver.unified_trips;